In [ ]:
import numpy as np
import json, os
import Volume.Selection as vs
import Data.IO as dio
import pandas as pd
import nibabel as ni

# Generate noSkull volume

In [ ]:
# Noskull volume
def noskull(main_dir):
    patients = sorted([os.path.join(main_dir, f, ses) for ses in os.listdir(sessions_dir) if "ses" in ses for f in os.listdir(main_dir) if "sub" in f]) if 'CHUV' in main_dir else sorted([os.path.join(main_dir, f) for f in os.listdir(main_dir) if "P" in f])
    for patient in patients:
        os.chdir(patient)
        if not os.path.isfile('noskull.nii.gz'):
            with open('config.json', 'r') as f:
                d=json.load(f)
                
            init_vol, vox2met = dio.read_nii_from_file(d['init volume'])
            
            mask = vs.removeSkullMask(init_vol)
            vol = mask*init_vol
            dio.save_nii_to_file('noskull.nii.gz', vol, vox2met)

main_dir = "/path/to/Data/CHUV/Voxel_annotation/"
noskull(main_dir)

## Extract Points

In [ ]:
r = 44
nbPoints = 30

In [ ]:
def extractPoints(patient_path, distance, nbPoints=30, out_file="points_new", overwrite=False):
    os.chdir(patient_path)
    patient_name = patient_path.split("/")[-1] if "P" in patient_path else "_".join(patient_path.split("/")[-2:])
    print(patient_name, end=": ")
    if (not overwrite) and os.path.isfile(out_file+".csv"):
        print("Already existing")
        return
        
    vol, vox2met = dio.read_nii_from_file('noskull.nii.gz')

    max_val = int(96/4)
    vals = list(range(0, max_val, 1)) + list(range(-1, -max_val-1, -1))
    vol[:, :, vals ] = 0
    vol[:, vals, : ] = 0
    vol[vals, :, : ] = 0

    fp = None if not os.path.isfile('F.csv') else dio.points_to_spheres(dio.read_points_from_csv('F.csv'))[:,:-1] # drop centers

    T = np.percentile(vol[vol>0], 95)
    Tl = np.percentile(vol[vol>0], 50)
    Th = np.percentile(vol[vol>0], 90)
    p  = vs.selectPoints(vol, vox2met, thresLow=T, r=35, fPoints=fp, nbPoints=nbPoints, extractType='Vessels')

    ex = p if fp is None else np.vstack((fp,p))

    q = vs.selectPoints(vol, vox2met, thresLow=Tl, thresHigh=Th, r=distance, fPoints=ex, nbPoints=nbPoints//2, extractType='Parenchyma')
    print(f'{len(p)} Vessel + {len(q)} Parenchyma')

    # export to csv using pandas
    ps = pd.DataFrame(p,columns=list('xyz'))
    ps['type'] = pd.Categorical(['Vessel']*len(p))
    qs=pd.DataFrame(q,columns=list('xyz'))
    qs['type'] = pd.Categorical(['Parenchyma']*len(q))

    points = ps.append(qs)
    points.to_csv(f'{out_file}.csv')
    dio.csv2fcsv (f'{out_file}.csv')

In [ ]:
main_dir = "/path/to/Data/CHRU/"
patients = sorted([os.path.join(main_dir, f, ses) for ses in os.listdir(sessions_dir) if "ses" in ses for f in os.listdir(main_dir) if "sub" in f]) if 'CHUV' in main_dir else sorted([os.path.join(main_dir, f) for f in os.listdir(main_dir) if "P" in f])
for patient in patients:
    extractPoints (patient, distance=r, nbPoints=nbPoints, overwrite=False)

## Check/count extracted negative patch centers (points.csv)

In [ ]:
main_dir = "/path/to/Data/CHRU/"
for patient in sorted([os.path.join(main_dir, f) for f in os.listdir(main_dir) if "P" in f]):
    os.chdir(patient)
    print(patient.split("/")[-1], end=": ")
    count = dio.read_points_from_csv("points_new.csv").shape[0]
    print(count, "points")

# Fcsv to csv files

In [ ]:
# CHRU & ADAM
main_dir = "/path/to/Data/CHRU/"
for patient in sorted([os.path.join(main_dir, f) for f in os.listdir(main_dir) if "P" in f]):
    os.chdir(patient)
    if os.path.isfile("F.csv")
        dio.fcsv2csv("F.fcsv")      

In [ ]:
# CHUV
main_dir = "/path/to/Data/CHUV/Sphere_annotation/"
patients = sorted([os.path.join(main_dir, f) for f in os.listdir(main_dir) if "sub" in f])
for sessions_dir in patients:
    sessions = [os.path.join(sessions_dir, f) for f in os.listdir(sessions_dir) if "ses" in f]
    for session in sessions:
        os.chdir(session)
        if os.path.isfile("F.csv")
            dio.fcsv2csv("F.fcsv")    

## Count aneurysms and get diameter

In [ ]:
def count_aneurysms(main_dir):
    CHUV = True if 'CHUV' in main_dir else False
    if not CHUV:
        patients = sorted([f for f in os.listdir(main_dir) if "." not in f and "P" in f])
        for patient in patients:
            fcsv_path = os.path.join(main_dir, patient, "F.fcsv")
            if os.path.isfile(fcsv_path):
                #dio.fcsv2csv (fcsv_path)
                print(patient, end =": ")
                points = dio.read_points_from_csv(fcsv_path.replace("fcsv", "csv"))
                for i in range(0, points.shape[0], 2):
                    p1, p2 = points[i], points[i+1]
                    size = np.linalg.norm(p1 - p2, axis=0)
                    print(round(size, 3), end=", ")
                print("")        
            #return
        
    else: # CHUV dataset
        patients = sorted([f for f in os.listdir(main_dir) if "sub" in f])
        print(len(patients))
        for patient in patients:
            sessions_dir = os.path.join(main_dir, patient)
            sessions = [f for f in os.listdir(sessions_dir) if "ses" in f]
            for session in sessions:
                session_dir = os.path.join(sessions_dir, session)
                print(patient, "->", session, end =": ")
                fcsv_path = os.path.join(session_dir, "F.fcsv")
                if os.path.isfile(fcsv_path):
                    # genetate csv from fcsv
                    #dio.fcsv2csv (fcsv_path)
                    # count aneurysms

                    points = dio.read_points_from_csv(fcsv_path.replace("fcsv", "csv"))
#                    if points.shape[0]/2 == 3: print("yes")
                    for i in range(0, points.shape[0], 2):
                        p1, p2 = points[i], points[i+1]
                        size = np.linalg.norm(p1 - p2, axis=0)
                        print(round(size, 3), end=", ")
                print("")


        
main_dir = "/path/to/Data/CHUV/Sphere_annotation//"
count_aneurysms(main_dir)

# Generate Truth masks

In [ ]:
# generate truth files from .Fcsv aneurysms files
from Volume.Edition import drawSphere
import numpy as np

main_dir = "/path/to/Data/CHUV/healthy_patients/Not_used/"
patients = sorted([os.path.join(main_dir, f, ses) for ses in os.listdir(sessions_dir) if "ses" in ses for f in os.listdir(main_dir) if "sub" in f]) if 'CHUV' in main_dir else sorted([os.path.join(main_dir, f) for f in os.listdir(main_dir) if "P" in f])
for patient in patients:
    patient_name = patient.split("/")[-1]
    print(patient_name, end=": ")
    os.chdir(patient)
    with open("config.json", 'r') as f:
        d = json.load(f)
    if os.path.isfile("mask_spheres.nii.gz"):
        print("Already generated")
    else:
        # load volume
        vol, vox2met = dio.read_nii_from_file (d['init volume'])
        t = np.zeros(vol.shape).astype(np.uint8)
        
        # load Fcsv points
        if d['pts aneurysm'] is not None and os.path.isfile(d['pts aneurysm']):
            spheres = dio.points_to_spheres(dio.read_points_from_csv (d['pts aneurysm']))
            i=-1
            for i, s in enumerate(spheres):
                drawSphere(t, vox2met, s[:3], s[3], val=1)
            print(i+1, 'aneurysms')
        dio.save_nii_to_file ("mask_spheres.nii.gz", t, vox2met)
        


# Generate N4 bias field correction

In [ ]:
import SimpleITK as sitk
import os, json
from tqdm import tqdm

In [ ]:
def generateCorrectedVolume(image_path, noskull_path, out_file):
    inputImage = sitk.ReadImage(image_path, sitk.sitkFloat32)
    maskImage = sitk.ReadImage(noskull_path)
    #maskImage, _ = dio.read_nii_from_file(noskull_path)
    #maskImage[maskImage>0] = 1
    maskImage = sitk.BinaryNot(sitk.BinaryThreshold(maskImage, 0, 0))
    #print(sitk.GetArrayFromImage(inputImage).shape, "vs", sitk.GetArrayFromImage(maskImage).shape)
    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    corrected_image = corrector.Execute(inputImage, maskImage)
    
    corrected_image_full_resolution = corrected_image
    
    #log_bias_field = corrector.GetLogBiasFieldAsImage(inputImage)
    #corrected_image_full_resolution = inputImage #/ sitk.Exp( log_bias_field )
    sitk.WriteImage(corrected_image_full_resolution, out_file)


In [ ]:
main_dir = "/path/to/Data/CHUV/Sphere_annotation/"
patients = sorted([f for f in os.listdir(main_dir) if "sub" in f])
print(len(patients))

In [ ]:
loop = tqdm(patients)
for patient in loop:
    sessions_dir = os.path.join(main_dir, patient)
    sessions = [f for f in os.listdir(sessions_dir) if "ses" in f]
    for session in sessions:
        session_dir = os.path.join(sessions_dir, session)
        os.chdir(session_dir)
    
        loop.set_description(f"{patient} -> {session}")
        with open("config.json", 'r') as f:
            config = json.load(f)
        if not os.path.isfile('N4biasCorrected.nii.gz'):
            #try:
            generateCorrectedVolume( config['init volume'], config['noskull volume'], 'N4biasCorrected.nii.gz')
            #except:
            #    print(patient)
        # append to json file
        config['N4 corrected volume'] = 'N4biasCorrected.nii.gz'
        with open('config.json', 'w') as f:
            json.dump(config, f, indent=4)


# Other stuff

In [ ]:
basic = [2.928, 2.483, 4.671, 2.527, 2.398, 5.408, 4.744, 6.444, 3.086, 2.959, 1.325, 6.124, 3.206, 3.229, 2.323, 1.499, 3.6, 5.102, 3.29, 8.068, 4.32, 5.361, 3.525, 4.356, 4.618, 5.376, 4.713, 1.624, 2.643, 2.697, 4.16, 2.142,]
unique = [3.06, 3.039, 1.996, 4.025, 2.659, 1.933, 4.537, 1.747, 4.555, 2.109, 4.719, 1.9, 3.093, 3.652, 2.402, 5.315, 1.783, 2.962, 4.932, 3.864, 2.572, 1.223, 5.3, 1.84, 2.268, 2.12, 5.233, 1.908, 3.689, 1.774, 2.543, ]
sizes = basic+unique

import statistics
print(len(sizes))
print(statistics.mean(sizes))
print(statistics.stdev(sizes))
print(min(sizes))
print(max(sizes))
print(len([f for f in sizes if f<3 ]))
print(len([f for f in sizes if f>=3 and f<5]))
print(len([f for f in sizes if f>=3])         )   

In [ ]:
sizes_chru = [9.241, 6.612, 4.079, 11.941, 4.064, 1.742, 5.236, 3.875, 2.412, 3.85, 3.374, 2.588, 3.719, 7.552, 5.363, 6.517, 3.102, 7.334, 6.638, 6.105, 9.354, 7.093, 4.833, 6.962, 1.962, 3.625, 3.601, 3.017, 5.854, 3.0, 7.528, 2.43, 6.068, 5.045, 8.095, 1.864, 2.746, 4.99, 1.678, 4.187, 1.744, 1.315, 1.319, 4.119, 2.771, 3.551, 2.692, 4.091, 5.032, 2.212, 2.121, 3.319, 3.209, 5.591, 3.195, 2.326, 2.799, 2.495, 2.381, 1.604, 1.315, 1.469, 3.903, 2.503, 3.808, 4.954, 5.603, 1.191, 3.815, 4.37, 3.014, 2.229, 5.557, 4.006, 5.163, 2.84, 2.225, 3.571, 4.432, 3.565, 3.323, 5.199, 3.674, 1.946, 1.907, 2.305, 4.432, 3.62, 2.042, 8.756, 1.979, 2.841, 2.465, 8.211, 3.274, 3.994, 4.676, 2.143, 6.727, 10.525, 2.974, 2.421, 6.488, 5.707, 2.987, 1.972, 4.425, 2.356, 1.516, 2.296, 1.659, 2.186, 2.445, 5.882, 2.807, 5.079, 1.47, 1.539, 6.169, 3.288, 4.904, 4.015, 2.556, 4.706, 3.833, 2.63, 3.86, 4.43, 4.314, 4.832, 1.311, 0.965, 3.914, 2.755, 6.496, 3.598, 4.906, 4.278, 2.347, 15.582, 2.829, 1.007, 2.04, 5.714, 7.623, 6.427, 2.409, 2.744, 3.921, 2.002, 2.877, 2.752, 1.364, 4.886, 4.1, 6.207, 1.775, 1.734, 3.075, 3.02, 3.394, 3.274, 5.573, 3.773, 2.562, 1.504, 2.855, 3.274, 3.48, 19.634, 3.854, 3.209, 2.906, 3.004, 4.763, 2.465, 7.505, 4.646, 3.171, 2.432, 1.956, 2.398, 4.388, 4.726, 3.978, 2.835, 4.027, 3.268, 2.261, 1.993, 2.659, 5.219, 4.199, 3.432, 6.155, 3.222, 5.548, 3.207, 2.852, 5.27, 2.85, 2.689, 5.126, 4.112, 8.775, 2.635]
# aneurysms_per_patient
#sizes_adam = []
sizes_chruv = [3.499, 2.658, 4.415, 3.844, 6.488, 2.016, 4.004, 6.102, 3.839, 4.137, 2.642, 4.553, 2.634, 3.06, 6.209, 1.928, 4.298, 2.632, 5.171, 7.759, 2.914, 3.569, 4.173, 2.787, 4.137, 1.735, 1.375, 3.468, 3.824, 2.567, 3.219, 3.971, 2.022, 2.972, 1.941, 3.317, 2.581, 2.082, 1.851] + [3.272, 5.766, 1.51, 3.06, 2.913, 3.28, 3.669, 3.018, 2.053, 2.927, 3.748, 4.89, 1.917, 3.225, 6.203, 3.857, 2.459, 12.908, 2.798, 6.382, 2.757, 1.233, 2.641, 1.617, 2.952, 5.604, 2.831, 2.041, 2.93, 5.83, 3.088, 1.721, 3.449, 3.032, 3.187, 1.549, 2.273, 4.661, 2.184, 4.544, 2.105, 2.63, 4.801, 2.719, 4.59, 3.238, 2.268, 2.044, 2.059, 4.624, 18.53, 13.645, 3.108, 3.325, 3.922, 5.086, 4.297, 5.17, 3.771, 7.19, 2.623, 2.578, 3.585, 1.901, 4.13, 4.251, 2.276, 2.553, 3.242, 3.02, 2.655, 3.168, 13.462, 6.731, 9.788, 3.357, 5.259, 4.498, 2.609, 3.747, 3.609, 2.057, 12.181, 2.374, 4.916, 3.404, 2.607, 2.642, 2.051, 2.071, 1.845, 3.396, 2.267, 3.863, 4.625, 3.014, 4.38, 3.931, 2.706, 5.152, 7.051, 2.711, 8.814, 3.963, 4.754, 3.928, 4.087, 5.325, 2.515, 2.701, 6.94, 3.221, 3.498, 2.664, 3.778, 5.166, 3.67, 2.124, 3.437, 4.692, 5.14, 2.206, 2.06, 4.537, 2.3, 3.108, 3.52, ]
print(len(sizes_chruv))
import statistics
print(statistics.mean(sizes_chruv))
print(statistics.stdev(sizes_chruv))
print(min(sizes_chruv))
print(max(sizes_chruv))
print(len([f for f in sizes_chruv if f<3 ]))
print(len([f for f in sizes_chruv if f>=3 and f<5]))
print(len([f for f in sizes_chruv if f>=3])         )   

In [ ]:
# generate 5-Fold Validation patient for CHUV dataset
import random
# get all patients path
main_dir = "/path/to/Data/CHUV/"
sub_dirs = ["healthy_patients", "Sphere_annotation", "Voxel_annotation"]

paths = []
for sub_dir in sub_dirs:
    #print("- ", sub_dir, ": ")
    tmp_dir = os.path.join(main_dir, sub_dir)
    patients = sorted([os.path.join (tmp_dir, f) for f in os.listdir(tmp_dir) if "sub" in f])
    for patient in patients:
        sessions = [os.path.join(patient, f) for f in os.listdir(patient) if "ses" in f]
        if len(sessions)>1: assert "Remove Follow-up session, please"
        paths.append(sessions[0])

random.shuffle(paths)

In [ ]:
nb_folds = 5
nb_paths_per_fold = int(len(paths)/nb_folds)

valid1 = paths[0: nb_paths_per_fold]
valid2 = paths[nb_paths_per_fold: nb_paths_per_fold*2]
valid3 = paths[nb_paths_per_fold*2: nb_paths_per_fold*3]
valid4 = paths[nb_paths_per_fold*3: nb_paths_per_fold*3*4]
valid5 = paths[nb_paths_per_fold*4:]

train1 = [f for f in paths if f not in valid1]
train2 = [f for f in paths if f not in valid2]
train3 = [f for f in paths if f not in valid3]
train4 = [f for f in paths if f not in valid4]
train5 = [f for f in paths if f not in valid5]

In [ ]:
print("Fold1: ", round((len([f for f in train1 if 'healthy_patients' in f])/ len(train1))*100, 3))
print("Fold2: ", round((len([f for f in train2 if 'healthy_patients' in f])/ len(train2))*100, 3))
print("Fold3: ", round((len([f for f in train3 if 'healthy_patients' in f])/ len(train3))*100, 3))
print("Fold4: ", round((len([f for f in train4 if 'healthy_patients' in f])/ len(train4))*100, 3))
print("Fold5: ", round((len([f for f in train5 if 'healthy_patients' in f])/ len(train5))*100, 3))

In [ ]:
# save splits to file
from Data.IO import saveSplit
saveSplit("/path/to/0Work/BaseConfig/CHUV/fold1.json", train1, [], valid1)
saveSplit("/path/to/0Work/BaseConfig/CHUV/fold2.json", train2, [], valid2)
saveSplit("/path/to/0Work/BaseConfig/CHUV/fold3.json", train3, [], valid3)
saveSplit("/path/to/0Work/BaseConfig/CHUV/fold4.json", train4, [], valid4)
saveSplit("/path/to/0Work/BaseConfig/CHUV/fold5.json", train5, [], valid5)

In [ ]:
# For ADAM
# generate 5-Fold Validation patient for CHUV dataset
import random
# get all patients path
main_dir = "/path/to/Data/ADAM/"

paths = []
tmp_dir = main_dir
patients = sorted([os.path.join (tmp_dir, f) for f in os.listdir(tmp_dir) if "P" in f])
for patient in patients:
    sessions = [os.path.join(patient, f) for f in os.listdir(patient) if "ses" in f]
    if len(sessions) > 1: assert "Remove Follow-up session, please"
    paths.append(sessions[0])

random.shuffle(paths)

# check treated aneurysms

In [ ]:
import Data.IO as dio
import os 
dir = "/path/to/Data/ADAM/"
def check (dir):
    patients = sorted([f for f in os.listdir(dir) if f[0]!="." and "png" not in f])
    for patient in patients:
        os.chdir(os.path.join(dir, patient))
        vol, vox2met = dio.read_nii_from_file('aneurysms.nii.gz')
        print(patient, vol[vol==2].sum())
    
check(dir)
